In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


In [2]:
# 資料獲取
# 取得台積電 (2330.TW) 過去 5 年的每日資料

ticker = "2330.TW"
df = yf.download(ticker, start="2019-01-01", end="2024-01-01")

# 只取收盤價，並重設索引讓日期變成一欄，方便像你原本的程式一樣操作
df = df[['Close']].reset_index()
print("資料前五筆：")
print(df.head())

[*********************100%***********************]  1 of 1 completed

資料前五筆：
Price        Date       Close
Ticker                2330.TW
0      2019-01-02  185.448700
1      2019-01-03  182.069168
2      2019-01-04  175.732666
3      2019-01-07  179.957031
4      2019-01-08  178.267273


In [ ]:
# --- 2. 資料整理 ---
# 建立落後項 (Lag features)
df['t-1'] = df['Close'].shift(1)
df['t-2'] = df['Close'].shift(2)
df['t-3'] = df['Close'].shift(3)

# 刪除因為 shift 產生的 NaN (這是你原本程式碼中最常卡住的地方)
df = df.dropna()

# 定義特徵 X 與目標 y
X = df[['t-1', 't-2', 't-3']]
y = df['Close'].iloc[:, 0] if isinstance(df['Close'], pd.DataFrame) else df['Close']

# 拆分訓練集與測試集 (保留 20% 測試)
# 注意：時間序列通常不隨機打亂，以維持時間順序
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [4]:
# 定義評估函式

def get_metrics(y_true, y_pred, name):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"[{name}] MAE: {mae:.2f}, RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")

In [5]:
# 線性回歸

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

get_metrics(y_test, lr_pred, "線性回歸")

[線性回歸] MAE: 5.32, RMSE: 7.28, MAPE: 1.03%


In [6]:
# 隨機森林

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

get_metrics(y_test, rf_pred, "隨機森林")

# --- 4. 視覺化預測結果 ---
plt.figure(figsize=(14, 7))
plt.plot(y_test.values, label='Actual Price', color='black', alpha=0.3)
plt.plot(lr_pred, label='Linear Regression', color='blue', linestyle='--')
plt.plot(rf_pred, label='Random Forest', color='red')
plt.title(f"{ticker} Price Prediction Comparison")
plt.legend()
plt.show()

c:\Users\user\miniconda3\envs\tsa_env\Lib\site-packages\sklearn\base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


ValueError: Unable to coerce to Series, length must be 1: given 243